In [116]:
import os
import certifi
import requests
from langchain.tools import tool

from dotenv import load_dotenv

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain_google_genai import ChatGoogleGenerativeAI


In [117]:
from langchain.agents import create_react_agent, AgentExecutor

In [118]:
# Load environment variables from .env file
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

Google_API_KEY = os.getenv("Google_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHER_API_KEY = os.getenv("WEATHER_FORECAST_API_KEY")


In [119]:
search_tool = TavilySearchResults(max_results=2)

In [120]:
result = search_tool.invoke("What is the captial of Frence ")
result

[{'url': 'https://www.britannica.com/place/France',
  'content': 'The capital of France is Paris. Situated in the north-central part of the country, Paris is France\'s center of commerce and culture.\n\nFor centuries, Paris has been one of the world’s most attractive cities. Nicknamed the "City of Light" during the Enlightenment, Paris is known for business, commerce, study, culture, and entertainment. The city is appreciated for its gastronomy, haute couture, painting, literature, and intellectual community. Paris is home to the Eiffel Tower, one of the world\'s top tourist attractions.\n\nFrance has played a central role in European culture for much of its history. French artistic, culinary, and sartorial styles have influenced cultures worldwide, and remain a point of national pride. [...] The capital and by far the most important city of France is Paris, one of the world’s preeminent cultural and commercial centres. A majestic city known as the ville lumière, or “city of light,” Pa

In [121]:
#LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=Google_API_KEY
)

In [122]:
response = llm.invoke("What year is it?, and what is the date today?, and latest news about gen z protest at jantar mantar")
response.content

"I do not have access to real-time information, live internet browsing, or current date and time data. Therefore, I cannot tell you today's date or provide the latest news regarding recent events."

In [135]:

@tool
def get_weather(city: str) -> str:
    """
    Get the current weather information for a given city using the Weatherstack API.

    Args:
        city (str): The name of the city to get weather information for.

    Returns:
        str: A human-readable current weather report.
    """

    if not WEATHER_API_KEY:
        return "Weatherstack API key is not configured."

    url = "https://api.weatherstack.com/current"

    params = {
        "access_key": WEATHER_API_KEY,
        "query": city
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        # Convert response to JSON first
        data = response.json()

        # Weatherstack API-level error
        if "error" in data:
                    error = data["error"]
        
                    error_code = error.get("code", "Unknown")
                    error_info = error.get("info", "Unknown Weatherstack error")
        
                    return (
                        f"Weatherstack API error.\n"
                        f"Error code: {error_code}\n"
                        f"Message: {error_info}"
                    )
        
        # HTTP-level error
        if response.status_code != 200:
            return (
                    f"Weather API request failed.\n"
                    f"HTTP status: {response.status_code}"
                )

        # Extract location information
        location = data.get("location", {})
        current = data.get("current", {})

        city_name = location.get("name", city)
        country = location.get("country", "")

        # Extract weather information
        temperature = current.get("temperature")
        feels_like = current.get("feelslike")
        humidity = current.get("humidity")
        condition = current.get(
            "weather_descriptions",
            ["Unknown"]
        )[0]

        wind_speed = current.get("wind_speed")
        wind_direction = current.get("wind_dir")

        pressure = current.get("pressure")
        visibility = current.get("visibility")
        precipitation = current.get("precip")

        return (
            f"Weather Report for {city_name}, {country}\n"
            f"Condition: {condition}\n"
            f"Temperature: {temperature}°C\n"
            f"Feels like: {feels_like}°C\n"
            f"Humidity: {humidity}%\n"
            f"Wind speed: {wind_speed} km/h\n"
            f"Wind direction: {wind_direction}\n"
            f"Pressure: {pressure} mb\n"
            f"Visibility: {visibility} km\n"
            f"Precipitation: {precipitation} mm"
        )

    except requests.exceptions.Timeout:
        return "Weather API request timed out."

    except requests.exceptions.ConnectionError:
        return "Could not connect to the Weatherstack API."

    except requests.exceptions.RequestException as e:
        return f"Weather API request failed: {str(e)}"

    except ValueError:
        return "Weatherstack returned an invalid JSON response."

    except Exception as e:
        return f"Unexpected error while getting weather: {str(e)}"

In [136]:
result = get_weather.invoke("New Delhi")

print(result)

Weather Report for New Delhi, India
Condition: Patchy rain nearby
Temperature: 36°C
Feels like: 42°C
Humidity: 48%
Wind speed: 10 km/h
Wind direction: SE
Pressure: 1001 mb
Visibility: 10 km
Precipitation: 0.1 mm


In [125]:
result = get_weather.invoke("New Delhi")
print(result)

Weather Report for New Delhi, India
Condition: Patchy rain nearby
Temperature: 36°C
Feels like: 42°C
Humidity: 48%
Wind speed: 10 km/h
Wind direction: SE
Pressure: 1001 mb
Visibility: 10 km
Precipitation: 0.1 mm


In [126]:
tools = [search_tool, get_weather]

In [127]:
prompt = hub.pull("hwchase17/react")
prompt

c:\Users\chand\OneDrive\Documents\Desktop\projects\AI-Agent practice\Langchain-agent\.venv\lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [128]:
#Agent
agent = create_react_agent(
    llm = llm, 
    tools = tools, 
    prompt = prompt)

In [129]:
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True
    )

In [132]:
response = agent_executor.invoke({
    "input" : ("What is the captial of Frence, and what is the year?,"
               " and latest news about gen z protest at jantar mantar"
               "weater at hyderbad"
               )
    })
print(response["output"])



> Entering new AgentExecutor chain...
Question: What is the captial of Frence, and what is the year?, and latest news about gen z protest at jantar mantarweater at hyderbad
Thought:
Action: tavily_search_results_json
Action Input: gen z protest jantar mantar latest news[{'url': 'https://en.wikipedia.org/wiki/2026_Delhi_Jantar_Mantar_protests', 'content': '| Indian Gen Z protests Cockroach protests Part of the Gen Z protests in Asia |\n|  |\n| Date | 6 June 2026 – 25 July 2026 (1 month and 19 days) |\n| Location | Jantar Mantar, New Delhi, India |\n| Caused by |  2026 NEET paper leak and related student suicides  CBSE On-Screen Marking irregularities  Repeated lapses in public examination administration over several decades, resulting in widespread paper leaks |\n| Goals |  Resignation of Union Education Minister Dharmendra Pradhan  Reforms in the examination system and the overall education system |\n| Methods | Demonstrations (Sansad Chalo march), sit-ins, hunger strikes, vandalism 